# CIS I — Assignment 3: Surface Registration with a Tracked Pointer

This notebook walks through the registration scenario of the handout
(Figure 1): a femur CT mesh, a reference body attached to the femur, a
tracked pointer, and an optical tracker. You **design** the pointer and
reference body and the camera placement, **collect** tip positions on
instructor-selected quasi-planar patches under a simulated clock, and
**register** the reference body to the CT mesh (`F_reg`, reference body → CT)
with a TRE of at most 2 mm, in reasonable simulated time and under 10 minutes
of wall-clock time.

The simulation (the femur's pose in the workspace, the true registration, the
pointer placement, the tracker and the simulated clock) runs on the **course
server**. Your code talks to it through a `Session` object
(`src/session.py`): `initial_registration_guess()`, `place_pointer(patch)`,
`sample_marker_bodies()`, `elapsed()`. The server only ever returns what the
tracker would report; nothing about the hidden truth leaves it. Grading runs
the same `Session` interface in-process on the instructor's machine with
different seeds.

Your code lives in `src/student/` (stubs raising `NotImplementedError`),
driven from here and unit-tested from `tests/`. The conventions are in the
README; read them first.

In [1]:
import sys, os, time
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
from src import vct3, Rot, Frame, load_sur, load_patch_boxes, RemoteSession, SceneDesign
from src.student import closest_point, icp, uncertainty, report, workflow

np.set_printoptions(precision=4, suppress=True)

DATA = os.path.join("..", "data")
mesh = load_sur(os.path.join(DATA, "ProximalFemurMeshFile.sur"))
boxes = load_patch_boxes(os.path.join(DATA, "box-centroids-rotations-world.json"))
print(f"mesh: {mesh.n_vertices} vertices, {mesh.n_triangles} triangles; {len(boxes)} patch boxes")

mesh: 1568 vertices, 3135 triangles; 46 patch boxes


## Step 1: Design the scene

Specify the tracker pose, the marker layouts of the reference body and the
pointer (both manufactured exactly), and the pointer tip `p_tip`. Keep in mind:

- The tracker's usable volume is a frustum along its **local +z axis**:
  100–220 cm in front of the camera, widening from a 100 cm to a 200 cm square.
  The femur sits near the workspace origin (±50 mm, ±0.3 rad).
- The pointer's +z axis points away from the bone when it touches the surface,
  so its markers should be on the +z side of the tip (e.g. `p_tip = (0, 0, -150)`
  with markers around the origin). Marker spread sets how noisy each pose is;
  the lever arms `‖p_tip‖` and the tip-to-reference distance multiply that noise.
- The reference body is attached close to the femur surface;
  `session.nominal_reference_pose()` gives its approximate workspace pose.

Put the same values in `workflow.design_scene()` — grading calls that function.

In [2]:
from src.student import design as scene_design

# Tracker 1600 mm from the femur along its optical axis (+z): femur at mid-depth of the frustum.
F_tracker_approx = scene_design.design_frame_tracker()
ref_marker_positions = scene_design.to_vct3_list(scene_design.REFERENCE_MARKERS_MM)   # reference-body coords (mm)
ptr_marker_positions = scene_design.to_vct3_list(scene_design.POINTER_MARKERS_MM)     # pointer coords (mm)
p_tip = vct3(*scene_design.POINTER_TIP_MM)                                           # tip, pointer coords (mm)

design = SceneDesign(F_tracker_approx, ref_marker_positions, ptr_marker_positions, p_tip)
assert vars(design).keys() == vars(workflow.design_scene()).keys()

# Predicted 1-sigma tip error of one pointer read for sigma_b = 0.1 mm (README) and 0.01 mm (handout)
for sigma_b in (0.1, 0.01):
    C_tip = scene_design.tracked_point_covariance(np.array(scene_design.POINTER_MARKERS_MM),
                                                  np.array(scene_design.POINTER_TIP_MM), sigma_b)
    print(f"sigma_b={sigma_b}: pointer tip rms error {np.sqrt(np.trace(C_tip)):.3f} mm per read")

sigma_b=0.1: pointer tip rms error 0.362 mm per read
sigma_b=0.01: pointer tip rms error 0.036 mm per read


## Step 2: Open a session on the course server

`SERVER_URL` and (if announced) `COURSE_TOKEN` are given on Piazza. A session
owns one hidden scene and its own simulated clock. Passing a `seed` makes a
practice session reproducible (same hidden truth and sample sequence);
grading uses other seeds and the same parameters. If the server is unreachable
you get a `RuntimeError` with the reason — check the URL and your network.

`SESSION_BACKEND` selects `"remote"` (course server, default) or `"offline"`
(our validation-only simulator in `validation/offline_session.py`, written from
the handout; use it when the server is unavailable). Set it here or with the
environment variable `A3_SESSION_BACKEND`.

In [3]:
from validation.offline_session import OfflineSession

SERVER_URL = "https://painting-classical-dosage-similar.trycloudflare.com"
COURSE_TOKEN = os.environ.get("A3_COURSE_TOKEN", "")    # paste the course token from piazza if required
SESSION_BACKEND = os.environ.get("A3_SESSION_BACKEND", "remote")
SEED = 1


def open_session(scene_design_values, seed):
    """Opens a practice session on the selected backend."""
    if SESSION_BACKEND == "remote":
        return RemoteSession(SERVER_URL, scene_design_values, seed=seed, token=COURSE_TOKEN)
    return OfflineSession(scene_design_values, mesh, boxes, seed=seed)


session = open_session(design, SEED)
F0_reg = session.initial_registration_guess()
print(f"session {session.session_id} ({session.mode}), {session.n_patches} patches, clock {session.elapsed():.2f} s")
print("F0_reg translation:", F0_reg.p.vec.ravel())

session 0ad7d96676f2fbe1 (practice), 46 patches, clock 0.00 s
F0_reg translation: [25. 85.  0.]


### Step 2.1: Visibility check

Before spending simulated time, check (locally, for free) that the reference
body and the pointer at every patch stay inside the frustum, using the nominal
poses the server returned.

In [4]:
ref_pts = [session.nominal_reference_pose() * a for a in ref_marker_positions]
print("reference body visible:", session.is_visible(ref_pts))
invisible = [i for i in range(session.n_patches)
             if not session.is_visible([session.nominal_pointer_pose(i) * a for a in ptr_marker_positions])]
print(f"pointer invisible on {len(invisible)}/{session.n_patches} patches: {invisible}")

reference body visible: True
pointer invisible on 0/46 patches: []


## Step 3: Unit tests

`tests/test_student.py` exercises your code offline (it needs no server):
complete tests fail with `NotImplementedError` until you implement what they
call; the `TODO` ones are yours to write.

In [5]:
!{sys.executable} -m pytest ../tests -q -p no:cacheprovider

......................                                                   [100%]
22 passed in 12.63s


## Step 4: Closest point on the mesh

Implement `closest_point.closest_point_on_triangle` and the vectorised
`closest_point.closest_points_on_mesh`. Time it: ICP calls it every iteration
on all your samples.

In [6]:
rng = np.random.default_rng(0)
q = rng.uniform(-60, 60, size=(200, 3))
t0 = time.perf_counter()
closest, tri, dist = closest_point.closest_points_on_mesh(q, mesh)
print(f"200 queries in {time.perf_counter() - t0:.3f} s; mean distance {dist.mean():.2f} mm")

200 queries in 0.016 s; mean distance 23.38 mm


## Step 5: Collect samples

Design `workflow.plan_patches` (which patches, how many samples each — the
residual statistics and the simulated time both depend on it) and
`workflow.collect_samples`: `session.place_pointer(patch)` (1 s simulated),
`session.sample_marker_bodies()` (0.03 s), then
`d_k = F_ref^-1 · F_ptr · p_tip` in reference-body coordinates. A body outside
the frustum comes back as `None` and still costs the time. Each call is one
HTTP round trip, so a few hundred samples take a few seconds of real time.

In [7]:
candidates = workflow.visible_boxes(session, boxes)
plan = workflow.plan_patches(candidates, n_patches=workflow.DEFAULT_N_PATCHES,
                             samples_per_patch=workflow.DEFAULT_SAMPLES_PER_PATCH)
print("planned patches (greedy TRE-optimal order):", list(dict.fromkeys(plan)))
samples = workflow.collect_samples(session, plan)
print(f"{len(samples)} samples from {len(set(samples.patch_ids))} patches in {samples.t_collect:.1f} s simulated, "
      f"{len(samples.skipped)} skipped")

planned patches (greedy TRE-optimal order): [24, 36, 29, 38, 37, 8, 23, 22, 4, 1, 43, 11, 33, 3, 12, 2]
80 samples from 16 patches in 87.2 s simulated, 0 skipped


## Step 6: Coarse registration and ICP

`F0_reg` is only a rough guess, so start with `icp.coarse_registration` (patch
sample centroids ↔ box centres, and the box planes), then `icp.icp` with your
outlier rejection. Explain both choices in the report.

In [8]:
d = samples.as_array()
F_coarse = icp.coarse_registration(samples.by_patch(), boxes)
result, start = icp.best_icp(d, mesh, [F_coarse, F0_reg])
print(f"ICP from {'coarse' if start == 0 else 'F0_reg'}: {result.n_iter} iterations, converged={result.converged}, "
      f"{result.inlier_mask.sum()}/{len(d)} samples used")
print("residual rms per iteration:", np.round(result.history, 3))

ICP from coarse: 7 iterations, converged=True, 66/80 samples used
residual rms per iteration: [2.396 0.523 0.169 0.13  0.127 0.127 0.127]


## Step 7: Uncertainty and the required report

Estimate `η_reg ~ N(μ, C_reg)` from the final fit and predict the TRE over
the mesh vertices (`uncertainty.py`), then assemble a `RegistrationReport`
(`workflow.run_registration` must do all of this headlessly).

In [9]:
mu, C_reg = uncertainty.registration_covariance(result, F0_reg)
pred = uncertainty.predicted_tre(mesh, result.F_reg, C_reg)
print("mu:", mu)
print("predicted TRE rms (mm):", pred["rms"])

mu: [ 0.3668  0.3639 -0.1953  1.4709 -2.4953 -5.3822]
predicted TRE rms (mm): 0.07797714661319402


## Step 8: Validation against the server's ground truth (practice sessions only)

The server never reveals `F*_reg`, but a practice session will score an
estimate for you: `session.evaluate_tre(F_reg)` returns TRE statistics over the
mesh vertices and `session.true_outliers()` the indices of the samples that
were outliers. Use them only here, for validation and your report: compare
achieved TRE vs predicted TRE vs residual statistics (FRE), and flagged vs
true outliers. Repeat for at least one other `SEED` and put the tables in
your report. Graded sessions refuse these calls.

In [10]:
tre = session.evaluate_tre(result.F_reg)                     # ground truth, reporting only
print(f"TRE over {tre.n} vertices: rms {tre.rms:.3f}  max {tre.max:.3f}  p90 {tre.p90:.3f} mm "
      f"(handout target 1 mm, README 2 mm)")
print(f"TRE of F0_reg {session.evaluate_tre(F0_reg).rms:.2f} mm, of the coarse registration {session.evaluate_tre(F_coarse).rms:.2f} mm")
true_outliers = set(session.true_outliers())
flagged = {h.sample_index for h, m in zip(samples.handles, result.inlier_mask) if not m}
print(f"true outliers {len(true_outliers)}, flagged {len(flagged)}, "
      f"correct {len(true_outliers & flagged)}, false alarms {len(flagged - true_outliers)}")
print(f"total simulated time {session.elapsed():.1f} s")

TRE over 1568 vertices: rms 0.063  max 0.105  p90 0.090 mm (handout target 1 mm, README 2 mm)
TRE of F0_reg 20.55 mm, of the coarse registration 3.63 mm
true outliers 14, flagged 14, correct 14, false alarms 0
total simulated time 87.2 s


## Achieved performance

Run the full headless pipeline once more on a fresh session (this is exactly
what grading calls) and print the report. Put the same tables in your written
report, for more than one seed.

In [11]:
session.close()

rows = []
for seed in (SEED, 2, 3, 4, 5):
    session = open_session(workflow.design_scene(), seed)
    final = workflow.run_registration(session, mesh, boxes)
    tre = session.evaluate_tre(final.F_reg)                  # practice only, after the headless run
    rows.append((seed, tre.rms, tre.max, final.predicted_tre["rms"], final.residual_stats["mean"],
                 final.residual_stats["p90"], final.t_simulated, final.t_wall))
    if seed == SEED:
        print(final.to_text())
        final.to_json("registration_report.json")
    session.close()

print("\nseed  TRE rms  TRE max  pred rms  FRE mean  FRE p90  t_sim(s)  t_wall(s)")
for r in rows:
    print(f"{r[0]:4d}  {r[1]:7.3f}  {r[2]:7.3f}  {r[3]:8.3f}  {r[4]:8.3f}  {r[5]:7.3f}  {r[6]:8.1f}  {r[7]:9.2f}")

F_reg (reference body -> CT):
     0.91703    0.24972    0.31095   45.59752
    -0.11976    0.91613   -0.38257   72.36687
    -0.38040    0.31359    0.87003   11.80099
     0.00000    0.00000    0.00000    1.00000

Patches (16):  id  samples  used  rejected
    1      5      5      0
    2      5      5      0
    3      5      5      0
    4      7      4      3
    8      5      5      0
   11      6      5      1
   12      6      5      1
   22      5      5      0
   23      5      5      0
   24      6      5      1
   29      6      5      1
   33      6      4      2
   36      6      4      2
   37      8      5      3
   38      7      5      2
   43      6      5      1

Valid samples (77): [0, 2, 3, 4, 5, 6, 7, 8, 10, 12, 13, 14, 16, 17, 19, 22, 23, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 42, 43, 45, 46, 47, 48, 49, 50, 51, 52, 54, 55, 57, 58, 59, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 75, 76, 77, 78, 79, 80, 82, 83, 84, 85, 86, 87, 88, 